# QuTech · QIG Hackathon 2026
## End-to-End Pipeline Walkthrough

> **Purpose:** This notebook walks through the complete hybrid classical + quantum  
> swaption-pricing pipeline — from raw data ingestion to training and evaluation.  
> It is self-contained and can be run in order, top to bottom.

---

### What you will see

| Section | Content |
|---|---|
| 1 · Environment | Dependency check and project imports |
| 2 · Data Ingestion | Load raw swaption price matrices |
| 3 · EDA | Shape, distributions, and term-structure heatmaps |
| 4 · Feature Engineering | Temporal features: lags, rolling stats, returns |
| 5 · Train / Val Split | Chronological split — zero data leakage |
| 6 · Classical Baseline | MLP training and metrics |
| 7 · Hybrid Quantum Model | MerLin encoder + MLP readout |
| 8 · Comparison | Side-by-side MAE / RMSE / R² |
| 9 · Visualizations | Loss curves, predicted vs actual, term surface |

---

> **Tip:** Cells marked `# CONFIG` at the top contain the parameters you are most  
> likely to want to change between runs.

---
## 1 · Environment Setup

We start by checking that the required packages are available and adding the  
project root to `sys.path` so all `src.*` modules resolve correctly.

In [ ]:
import sys, importlib, subprocess

# ── Add project root to path ───────────────────────────────────────────────────
sys.path.insert(0, "..")   # adjust if running from a sub-directory

# ── Dependency check ───────────────────────────────────────────────────────────
REQUIRED  = ["numpy", "pandas", "matplotlib", "seaborn"]
OPTIONAL  = {"torch": "model training", "sklearn": "Ridge baseline + StandardScaler"}

missing_required = []
for pkg in REQUIRED:
    try:
        importlib.import_module(pkg)
        print(f"  ✅  {pkg}")
    except ImportError:
        print(f"  ❌  {pkg}  ← REQUIRED — run: pip install {pkg}")
        missing_required.append(pkg)

for pkg, purpose in OPTIONAL.items():
    try:
        importlib.import_module(pkg)
        print(f"  ✅  {pkg}  ({purpose})")
    except ImportError:
        print(f"  ⚠️   {pkg}  ({purpose}) — optional, some cells will be skipped")

if missing_required:
    raise EnvironmentError(f"Install missing packages before continuing: {missing_required}")
else:
    print("\n🟢  Environment ready.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings, os, json
from pathlib import Path

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110

# ── Check for torch ────────────────────────────────────────────────────────────
try:
    import torch
    TORCH_AVAILABLE = True
    print(f"PyTorch {torch.__version__} — device: ", end="")
    if torch.cuda.is_available():
        print(f"CUDA ({torch.cuda.get_device_name(0)})")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("MPS (Apple Silicon)")
    else:
        print("CPU")
except ImportError:
    TORCH_AVAILABLE = False
    print("⚠️  PyTorch not found — training cells will be skipped")

---
## 2 · Data Ingestion

`load_data()` in `src/data/loader.py` supports `.xlsx`, `.csv`, and  
`s3://bucket/prefix` paths. Here we load from the local `DATASETS/` folder.

The raw file is a **wide-format** matrix where each column represents a  
different maturity bucket for a given tenor — e.g. `Swaption_1Y_3M`.

In [ ]:
# CONFIG ──────────────────────────────────────────────────────────────────────
DATA_DIR   = "../DATASETS"        # path to the DATASETS folder
TRAIN_FILE = "train.xlsx"         # training file name
TEST_FILE  = "test_template.xlsx" # test template

# ─────────────────────────────────────────────────────────────────────────────
try:
    from src.data.loader import load_train_data, load_test_template
    df_raw  = load_train_data(DATA_DIR, TRAIN_FILE)
    df_test = load_test_template(DATA_DIR, TEST_FILE)
    print(f"Train loaded  : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} cols")
    print(f"Test template : {df_test.shape[0]:,} rows × {df_test.shape[1]} cols")
except Exception as e:
    # Graceful fallback: generate synthetic data for demonstration
    print(f"⚠️  Could not load from src ({e})")
    print("   Generating synthetic swaption data for demonstration...")
    rng = np.random.default_rng(42)
    dates = pd.date_range("2040-01-01", periods=500, freq="B")
    tenors    = ["1Y", "2Y", "5Y", "10Y"]
    maturities = ["3M", "6M", "1Y", "2Y", "5Y"]
    cols = {f"Swaption_{t}_{m}": rng.lognormal(mean=-2, sigma=0.3, size=len(dates))
            for t in tenors for m in maturities}
    df_raw = pd.DataFrame({"Date": dates, **cols})
    df_test = df_raw.tail(50).copy()
    print(f"  Synthetic train : {df_raw.shape}")
    print(f"  Synthetic test  : {df_test.shape}")

In [ ]:
# Quick sanity-check on column names and date range
print("Date range :", df_raw["Date"].min(), "→", df_raw["Date"].max())
print("\nFirst 3 rows:")
df_raw.head(3)

---
## 3 · Exploratory Data Analysis

Before any feature engineering we inspect:
- **Missing values** — are there gaps in the time series?
- **Price distributions** — are the raw swaption prices log-normally distributed?
- **Term structure heatmap** — how does implied vol vary across tenor × maturity?

In [ ]:
# ── 3.1 Missing values ────────────────────────────────────────────────────────
missing = df_raw.isnull().sum()
n_missing = missing[missing > 0]
if n_missing.empty:
    print("✅  No missing values found.")
else:
    print("⚠️  Columns with missing values:")
    print(n_missing.to_string())

In [ ]:
# ── 3.2 Price distribution ────────────────────────────────────────────────────
price_cols = [c for c in df_raw.columns if c != "Date"]
all_prices = df_raw[price_cols].values.flatten()
all_prices = all_prices[~np.isnan(all_prices)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(all_prices, bins=60, color="#4C72B0", edgecolor="white", linewidth=0.4)
axes[0].set_title("Raw price distribution")
axes[0].set_xlabel("Swaption price")
axes[0].set_ylabel("Count")

axes[1].hist(np.log1p(all_prices), bins=60, color="#55A868", edgecolor="white", linewidth=0.4)
axes[1].set_title("Log(1 + price) distribution")
axes[1].set_xlabel("log(1 + price)")
axes[1].set_ylabel("Count")

fig.suptitle("Swaption price distributions across all tenor-maturity pairs", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.3 Term structure heatmap (snapshot at last available date) ──────────────
import re

# Parse tenor / maturity from column names like "Swaption_1Y_3M"
records = []
for col in price_cols:
    m = re.match(r"Swaption_(.+?)_(.+)", col)
    if m:
        records.append({"col": col, "tenor": m.group(1), "maturity": m.group(2)})

if records:
    meta = pd.DataFrame(records)
    last_row = df_raw.iloc[-1]
    meta["price"] = meta["col"].map(last_row)

    pivot = meta.pivot(index="tenor", columns="maturity", values="price")

    # Natural ordering heuristics
    order_map = {"3M":0,"6M":1,"9M":2,"1Y":3,"2Y":4,"3Y":5,"5Y":6,"7Y":7,"10Y":8,"15Y":9,"20Y":10,"30Y":11}
    pivot = pivot.reindex(sorted(pivot.columns, key=lambda x: order_map.get(x, 99)), axis=1)

    fig, ax = plt.subplots(figsize=(12, 4))
    sns.heatmap(pivot.astype(float), annot=True, fmt=".3f", cmap="YlOrRd",
                linewidths=0.5, ax=ax, cbar_kws={"label": "Price"})
    ax.set_title(f"Term structure snapshot — {str(last_row['Date'])[:10]}", fontsize=13)
    ax.set_xlabel("Maturity")
    ax.set_ylabel("Tenor")
    plt.tight_layout()
    plt.show()
else:
    print("Could not parse tenor/maturity from column names — skipping heatmap.")

In [ ]:
# ── 3.4 Price time series for a selection of tenor-maturity pairs ──────────────
sample_cols = price_cols[:min(5, len(price_cols))]

fig, ax = plt.subplots(figsize=(13, 4))
for col in sample_cols:
    ax.plot(df_raw["Date"], df_raw[col], label=col, linewidth=1.2)
ax.set_title("Swaption price time series — selected pairs", fontsize=13)
ax.set_xlabel("Date")
ax.set_ylabel("Price")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

---
## 4 · Temporal Feature Engineering

`build_temporal_dataset()` in `src/data/preprocessing.py` applies a four-step  
pipeline that transforms the wide-format matrix into a long-format time series  
enriched with temporal signals:

1. **`melt_maturities`** — wide → long: one row per (date, tenor, maturity)
2. **`add_temporal_features`** — lags, 1-step diff, log-return, rolling mean/std
3. **`normalize_prices`** — z-score normalisation (with sklearn fallback)
4. **`prepare_features`** — select and order final feature columns

The resulting feature matrix captures *how prices evolve over time*, which is  
critical for a model that must generalise to unseen future dates.

In [ ]:
# CONFIG ──────────────────────────────────────────────────────────────────────
LAGS            = [1, 5, 10]   # lag steps (trading days)
ROLLING_WINDOWS = [5, 20]      # rolling mean/std window sizes (trading days)

# ─────────────────────────────────────────────────────────────────────────────
try:
    from src.data.preprocessing import build_temporal_dataset
    df_feat = build_temporal_dataset(df_raw, lags=LAGS, rolling_windows=ROLLING_WINDOWS)
    print(f"Feature matrix : {df_feat.shape[0]:,} rows × {df_feat.shape[1]} columns")
    print(f"\nFeature columns ({len(df_feat.columns)}):")
    for i, c in enumerate(df_feat.columns):
        print(f"  {i+1:>3}. {c}")
except Exception as e:
    print(f"⚠️  src not available ({e})")
    print("   Building minimal temporal features manually for demonstration...")

    # Minimal manual implementation matching the pipeline behaviour
    price_col = price_cols[0]
    df_feat = df_raw[["Date", price_col]].copy().rename(columns={price_col: "price"})

    for lag in LAGS:
        df_feat[f"price_lag_{lag}"] = df_feat["price"].shift(lag)
    df_feat["price_diff_1"]   = df_feat["price"].diff(1)
    df_feat["price_return_1"] = np.log(df_feat["price"] / df_feat["price"].shift(1))
    for w in ROLLING_WINDOWS:
        df_feat[f"price_roll_mean_{w}"] = df_feat["price"].rolling(w).mean()
        df_feat[f"price_roll_std_{w}"]  = df_feat["price"].rolling(w).std()

    df_feat["time_idx_days"] = (df_feat["Date"] - df_feat["Date"].min()).dt.days

    # Normalise price
    mu, sigma = df_feat["price"].mean(), df_feat["price"].std()
    df_feat["price"] = (df_feat["price"] - mu) / (sigma + 1e-8)

    df_feat.dropna(inplace=True)
    df_feat.reset_index(drop=True, inplace=True)
    print(f"  Feature matrix : {df_feat.shape[0]:,} rows × {df_feat.shape[1]} columns")

In [ ]:
# Correlation heatmap of engineered features
feat_cols  = [c for c in df_feat.columns if c not in ("Date", "tenor", "maturity")]
corr_matrix = df_feat[feat_cols].corr()

fig, ax = plt.subplots(figsize=(min(14, len(feat_cols)), min(11, len(feat_cols)-1)))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap="coolwarm", center=0,
            annot=(len(feat_cols) <= 15), fmt=".2f",
            linewidths=0.4, ax=ax, cbar_kws={"shrink": 0.7})
ax.set_title("Feature correlation matrix", fontsize=13)
plt.tight_layout()
plt.show()

---
## 5 · Chronological Train / Validation Split

We split the data **strictly by time**: the last `VAL_FRACTION` of rows  
(ordered by date) become the validation set.  
This mirrors real-world deployment — the model never sees future data during  
training — and prevents any look-ahead bias.

In [ ]:
# CONFIG ──────────────────────────────────────────────────────────────────────
VAL_FRACTION = 0.2   # fraction of data reserved for validation
TARGET_COL   = "price"

# ─────────────────────────────────────────────────────────────────────────────
try:
    from src.data.splits import time_based_split
    train_df, val_df = time_based_split(df_feat, val_fraction=VAL_FRACTION)
except Exception:
    # Manual chronological split
    split_idx = int(len(df_feat) * (1 - VAL_FRACTION))
    train_df  = df_feat.iloc[:split_idx].copy()
    val_df    = df_feat.iloc[split_idx:].copy()

feature_cols = [c for c in df_feat.columns if c not in ("Date", "tenor", "maturity", TARGET_COL)]
X_train = train_df[feature_cols].values.astype(np.float32)
y_train = train_df[TARGET_COL].values.astype(np.float32)
X_val   = val_df[feature_cols].values.astype(np.float32)
y_val   = val_df[TARGET_COL].values.astype(np.float32)

split_date = val_df["Date"].min() if "Date" in val_df.columns else "N/A"
print(f"Split date   : {str(split_date)[:10]}")
print(f"Train set    : {X_train.shape[0]:,} rows  ({X_train.shape[1]} features)")
print(f"Val set      : {X_val.shape[0]:,} rows")
print(f"\nTarget stats (train) — mean: {y_train.mean():.4f}  std: {y_train.std():.4f}")

In [ ]:
# Visualise the split boundary
if "Date" in df_feat.columns:
    fig, ax = plt.subplots(figsize=(13, 3.5))
    ax.plot(train_df["Date"], y_train, color="#4C72B0", linewidth=1,   label=f"Train ({len(y_train):,} rows)")
    ax.plot(val_df["Date"],   y_val,   color="#C44E52", linewidth=1.2, label=f"Val ({len(y_val):,} rows)")
    ax.axvline(x=val_df["Date"].min(), color="gray", linestyle="--", linewidth=1, label="Split boundary")
    ax.set_title("Target price — train / validation split", fontsize=13)
    ax.set_xlabel("Date"); ax.set_ylabel("Normalised price")
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## 6 · Classical Baseline — MLP

`src/classical/mlp.py` provides a configurable PyTorch MLP.  
We train it here as our **performance floor**: any hybrid model that does not  
beat these numbers is not providing a useful quantum advantage.

Training uses:
- **Optimizer:** Adam
- **Loss:** Mean Squared Error (MSE)
- **Early stopping:** monitored on validation loss

In [ ]:
# CONFIG ──────────────────────────────────────────────────────────────────────
EPOCHS    = 60
LR        = 1e-3
BATCH     = 32
LOG_EVERY = 10

# ─────────────────────────────────────────────────────────────────────────────
if not TORCH_AVAILABLE:
    print("⚠️  PyTorch not available — skipping MLP training.")
else:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader

    def make_loaders(X_tr, y_tr, X_v, y_v, batch_size=32):
        to_t = lambda a: torch.tensor(a, dtype=torch.float32)
        tr = DataLoader(TensorDataset(to_t(X_tr), to_t(y_tr)), batch_size=batch_size, shuffle=True)
        vl = DataLoader(TensorDataset(to_t(X_v),  to_t(y_v)),  batch_size=batch_size)
        return tr, vl

    train_loader, val_loader = make_loaders(X_train, y_train, X_val, y_val, BATCH)
    print(f"DataLoaders ready — train batches: {len(train_loader)} | val batches: {len(val_loader)}")

In [ ]:
if TORCH_AVAILABLE:
    try:
        from src.classical.mlp import MLP
        model_mlp = MLP(input_dim=X_train.shape[1])
    except Exception:
        # Inline fallback MLP definition
        class MLP(nn.Module):
            def __init__(self, input_dim, hidden=[128, 64, 32]):
                super().__init__()
                layers, dims = [], [input_dim] + hidden
                for i in range(len(dims)-1):
                    layers += [nn.Linear(dims[i], dims[i+1]), nn.BatchNorm1d(dims[i+1]), nn.ReLU()]
                layers.append(nn.Linear(dims[-1], 1))
                self.net = nn.Sequential(*layers)
            def forward(self, x):
                return self.net(x).squeeze(-1)

        model_mlp = MLP(input_dim=X_train.shape[1])

    n_params = sum(p.numel() for p in model_mlp.parameters() if p.requires_grad)
    print(f"MLP architecture:")
    print(model_mlp)
    print(f"\nTrainable parameters: {n_params:,}")

In [ ]:
if TORCH_AVAILABLE:
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model_mlp.parameters(), lr=LR)

    history_mlp = {"train_loss": [], "val_loss": []}

    for epoch in range(1, EPOCHS + 1):
        # ── Train ──────────────────────────────────────────────────────────────
        model_mlp.train()
        tr_losses = []
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model_mlp(xb), yb)
            loss.backward()
            optimizer.step()
            tr_losses.append(loss.item())

        # ── Validate ───────────────────────────────────────────────────────────
        model_mlp.eval()
        vl_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                vl_losses.append(criterion(model_mlp(xb), yb).item())

        history_mlp["train_loss"].append(np.mean(tr_losses))
        history_mlp["val_loss"].append(np.mean(vl_losses))

        if epoch % LOG_EVERY == 0 or epoch == 1:
            print(f"  Epoch {epoch:>3}/{EPOCHS}  "
                  f"train loss: {history_mlp['train_loss'][-1]:.5f}  "
                  f"val loss: {history_mlp['val_loss'][-1]:.5f}")

    print("\n✅  MLP training complete.")

---
## 7 · Hybrid Quantum Model — MerLin Encoder + MLP Head

`src/hybrid/model.py` implements `MerlinHybridRegressor`:

```
input features  →  [Quantificator]  →  MerLin QuantumLayer  →  classical MLP head  →  price
```

The **Quantificator** (`src/quantum/encoder.py`) maps classical feature vectors  
to quantum-ready inputs using either:
- **Angle encoding** — features are mapped to Rx/Ry rotation angles on each mode  
- **Amplitude encoding** — features are loaded directly as probability amplitudes

If the MerLin backend is not installed, the pipeline automatically falls back  
to a **PyTorch-simulated quantum encoder** so training can still proceed.

In [ ]:
# CONFIG ──────────────────────────────────────────────────────────────────────
N_MODES       = 4
N_PHOTONS     = 2
QUANTUM_DEPTH = 2
ENCODING_TYPE = "angle"    # "angle" | "amplitude"
BACKEND       = "auto"     # "merlin" | "simulated" | "auto"

EPOCHS_HYB  = 60
LR_HYB      = 5e-4

# ─────────────────────────────────────────────────────────────────────────────
if not TORCH_AVAILABLE:
    print("⚠️  PyTorch not available — skipping hybrid model.")
else:
    try:
        from src.hybrid.model import MerlinHybridRegressor
        model_hyb = MerlinHybridRegressor(
            input_dim=X_train.shape[1],
            n_modes=N_MODES,
            n_photons=N_PHOTONS,
            quantum_depth=QUANTUM_DEPTH,
            encoding_type=ENCODING_TYPE,
            backend=BACKEND,
        )
        print(f"Backend selected  : {model_hyb.backend_name}")
    except Exception as e:
        print(f"⚠️  Could not load MerlinHybridRegressor ({e})")
        print("   Falling back to simulated hybrid model...")

        # Simulated quantum layer (cosine feature map)
        class SimulatedQuantumLayer(nn.Module):
            def __init__(self, in_dim, out_dim):
                super().__init__()
                self.linear = nn.Linear(in_dim, out_dim)
                self.out_dim = out_dim
            def forward(self, x):
                z = self.linear(x)
                return torch.cos(z) * torch.sin(z)  # non-linear quantum-inspired map

        class MerlinHybridRegressor(nn.Module):
            def __init__(self, input_dim, quantum_out=16, hidden=[64, 32]):
                super().__init__()
                self.encoder = SimulatedQuantumLayer(input_dim, quantum_out)
                layers, dims = [], [quantum_out] + hidden
                for i in range(len(dims)-1):
                    layers += [nn.Linear(dims[i], dims[i+1]), nn.ReLU()]
                layers.append(nn.Linear(dims[-1], 1))
                self.head = nn.Sequential(*layers)
                self.backend_name = "simulated (fallback)"
            def forward(self, x):
                return self.head(self.encoder(x)).squeeze(-1)

        model_hyb = MerlinHybridRegressor(input_dim=X_train.shape[1])
        print(f"Backend selected  : {model_hyb.backend_name}")

    n_params_hyb = sum(p.numel() for p in model_hyb.parameters() if p.requires_grad)
    print(f"Trainable parameters: {n_params_hyb:,}")

In [ ]:
if TORCH_AVAILABLE:
    criterion_hyb = nn.MSELoss()
    optimizer_hyb = torch.optim.Adam(model_hyb.parameters(), lr=LR_HYB)

    history_hyb = {"train_loss": [], "val_loss": []}

    for epoch in range(1, EPOCHS_HYB + 1):
        model_hyb.train()
        tr_losses = []
        for xb, yb in train_loader:
            optimizer_hyb.zero_grad()
            loss = criterion_hyb(model_hyb(xb), yb)
            loss.backward()
            optimizer_hyb.step()
            tr_losses.append(loss.item())

        model_hyb.eval()
        vl_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                vl_losses.append(criterion_hyb(model_hyb(xb), yb).item())

        history_hyb["train_loss"].append(np.mean(tr_losses))
        history_hyb["val_loss"].append(np.mean(vl_losses))

        if epoch % LOG_EVERY == 0 or epoch == 1:
            print(f"  Epoch {epoch:>3}/{EPOCHS_HYB}  "
                  f"train loss: {history_hyb['train_loss'][-1]:.5f}  "
                  f"val loss: {history_hyb['val_loss'][-1]:.5f}")

    print("\n✅  Hybrid model training complete.")

---
## 8 · Model Comparison

We evaluate both models on the held-out validation set using the shared  
metrics defined in `src/eval/metrics.py`:

| Metric | Formula | Interpretation |
|---|---|---|
| **MAE** | mean\|ŷ − y\| | Average absolute error in price units |
| **RMSE** | √mean(ŷ − y)² | Penalises large errors more heavily |
| **R²** | 1 − SS_res/SS_tot | Fraction of variance explained (1.0 = perfect) |

In [ ]:
def compute_metrics(y_true, y_pred):
    """MAE, RMSE, R² — works with numpy arrays."""
    try:
        from src.eval.metrics import mae, rmse, r2
        return {"MAE": mae(y_true, y_pred), "RMSE": rmse(y_true, y_pred), "R2": r2(y_true, y_pred)}
    except Exception:
        residuals = y_true - y_pred
        mae_  = np.mean(np.abs(residuals))
        rmse_ = np.sqrt(np.mean(residuals ** 2))
        ss_res = np.sum(residuals ** 2)
        ss_tot = np.sum((y_true - y_true.mean()) ** 2)
        r2_   = 1 - ss_res / (ss_tot + 1e-10)
        return {"MAE": mae_, "RMSE": rmse_, "R2": r2_}

results = {}

if TORCH_AVAILABLE:
    Xt = torch.tensor(X_val, dtype=torch.float32)

    model_mlp.eval()
    with torch.no_grad():
        preds_mlp = model_mlp(Xt).numpy()
    results["MLP Baseline"] = compute_metrics(y_val, preds_mlp)

    model_hyb.eval()
    with torch.no_grad():
        preds_hyb = model_hyb(Xt).numpy()
    results["Hybrid (MerLin)"] = compute_metrics(y_val, preds_hyb)

    df_results = pd.DataFrame(results).T
    df_results.index.name = "Model"
    print("\n── Validation metrics ──────────────────────────────")
    print(df_results.round(5).to_string())
else:
    print("⚠️  Metrics require PyTorch. Skipping.")

In [ ]:
if TORCH_AVAILABLE and results:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    metrics_list = ["MAE", "RMSE", "R2"]
    colors = ["#4C72B0", "#DD8452"]

    for ax, metric in zip(axes, metrics_list):
        values = [results[m][metric] for m in results]
        bars   = ax.bar(list(results.keys()), values, color=colors, edgecolor="white", linewidth=0.6)
        ax.set_title(metric, fontsize=13, fontweight="bold")
        ax.set_ylabel(metric)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                    f"{val:.4f}", ha="center", va="bottom", fontsize=9)

    fig.suptitle("Validation metrics — MLP Baseline vs Hybrid (MerLin)", fontsize=13)
    plt.tight_layout()
    plt.show()

---
## 9 · Visualisations

Three standard diagnostic plots for regression on time-series data:

1. **Loss curves** — confirms training convergence and checks for overfitting  
2. **Predicted vs actual** — reveals systematic biases or heteroscedasticity  
3. **Residual distribution** — should be centred at zero and roughly Gaussian

In [ ]:
if TORCH_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    epochs_ax = range(1, max(len(history_mlp["train_loss"]), len(history_hyb["train_loss"])) + 1)

    for ax, history, title, color in zip(
        axes,
        [history_mlp, history_hyb],
        ["MLP Baseline — loss curves", "Hybrid Model — loss curves"],
        ["#4C72B0", "#DD8452"]
    ):
        ep = range(1, len(history["train_loss"]) + 1)
        ax.plot(ep, history["train_loss"], label="Train",      color=color,        linewidth=1.5)
        ax.plot(ep, history["val_loss"],   label="Validation", color=color, alpha=0.5, linewidth=1.5, linestyle="--")
        ax.set_title(title, fontsize=12)
        ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss")
        ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
if TORCH_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for ax, preds, title in zip(
        axes,
        [preds_mlp, preds_hyb],
        ["MLP Baseline — predicted vs actual", "Hybrid Model — predicted vs actual"]
    ):
        ax.scatter(y_val, preds, alpha=0.4, s=18, edgecolors="none", color="#4C72B0")
        lims = [min(y_val.min(), preds.min()), max(y_val.max(), preds.max())]
        ax.plot(lims, lims, "r--", linewidth=1.2, label="Perfect prediction")
        ax.set_xlabel("Actual price (normalised)")
        ax.set_ylabel("Predicted price")
        ax.set_title(title, fontsize=12)
        ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
if TORCH_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    for ax, preds, title in zip(
        axes,
        [preds_mlp, preds_hyb],
        ["MLP Baseline — residuals", "Hybrid Model — residuals"]
    ):
        residuals = y_val - preds
        ax.hist(residuals, bins=40, color="#55A868", edgecolor="white", linewidth=0.4)
        ax.axvline(0, color="red", linestyle="--", linewidth=1.2, label="Zero residual")
        ax.set_xlabel("Residual (actual − predicted)")
        ax.set_ylabel("Count")
        ax.set_title(title, fontsize=12)
        ax.legend()
        ax.annotate(f"mean = {residuals.mean():.4f}\nstd  = {residuals.std():.4f}",
                    xy=(0.97, 0.95), xycoords="axes fraction",
                    ha="right", va="top", fontsize=9,
                    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))

    plt.tight_layout()
    plt.show()

---
## Summary

| | MLP Baseline | Hybrid (MerLin) |
|---|---|---|
| Architecture | Feedforward MLP | Quantum encoder + MLP head |
| Parameters | see Section 6 | see Section 7 |
| Quantum backend | — | MerLin / simulated |
| Metrics | see Section 8 | see Section 8 |

### Next steps
- Tune quantum circuit hyperparameters (`--n-modes`, `--n-photons`, `--quantum-depth`)
- Run config-driven experiments: `python run.py --config configs/hybrid.yaml`
- Generate the full technical report: `python src/data/technical_report.py`
- Submit predictions: `python predict_interface.py --checkpoint results/checkpoint.pt ...`

---
*Generated for QuTech · QIG Hackathon 2026*